# Electoral Modelling — General Result

Loads every input CSV for the General election model (`data/csv_input/General`) into dataframes. Same reusable building blocks as the Alma Vale notebook (`CrossTableMultiplier`, `GaussianRandomizer`, `random_no_generation.random_normal_adv`) will be layered on top once the chained/randomised calculations start.

In [27]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(r"E:\Coding Site\just_for_fun")
from random_no_generation import random_normal_adv

ELECTORAL_MODELLING_DIR = Path(r"E:\Coding Site\just_for_fun\Electoral_Modelling")
GENERAL_CSV_DIR = ELECTORAL_MODELLING_DIR / "data" / "csv_input" / "General"

## Load all CSVs in `csv_input/General`

In [28]:
dfs = {path.stem: pd.read_csv(path) for path in sorted(GENERAL_CSV_DIR.glob("*.csv"))}
df_13d = dfs["df_input_13D"]
df_general_direct = dfs["df_input_general_direct"]
df_general_proportional = dfs["df_input_general_proportional"]
df_generaldirect_referendum1 = dfs["df_input_generaldirect_referendum1"]

In [29]:
class GaussianRandomizer:
    """
    Draws Normal(mean=cell, sd=sd_fraction*cell) per cell via random_no_generation.random_normal_adv
    (mean == 0 cells are always left at 0 - no draw).

    Gate (optional, low/high): redraws only cells outside [low, high] - checked against the raw
    value, or against value/reference if `reference` is passed to randomize() (e.g. gate on
    turnout% = turnout / electorate) - up to max_attempts times.

    clamp_to_reference (randomize()): after gating, hard-clip any still-out-of-range cell to
    [0, reference] instead of leaving it as an outlier.
    """

    def __init__(self, sd_fraction=0.1, low=None, high=None, max_attempts=10, gate_enabled=True):
        self.sd_fraction = sd_fraction
        self.low = low
        self.high = high
        self.max_attempts = max_attempts
        self.gate_enabled = gate_enabled

    def randomize(self, means_df, reference=None, sd=None, clamp_to_reference=False):
        means = means_df.to_numpy(dtype=float)
        sds = self._resolve_sd(means, sd)
        ref = reference.to_numpy(dtype=float) if reference is not None else None

        values = self._draw(means, sds)
        attempts_used, unresolved = 1, 0
        if self.gate_enabled and (self.low is not None or self.high is not None):
            values, attempts_used, unresolved = self._gate_and_redo(values, means, sds, ref)

        if clamp_to_reference and ref is not None:
            values = np.clip(values, 0, ref)

        result = pd.DataFrame(values, columns=means_df.columns, index=means_df.index)
        return result, {"attempts_used": attempts_used, "unresolved_cells": unresolved}

    def _resolve_sd(self, means, sd):
        if sd is None:
            return means * self.sd_fraction
        if np.isscalar(sd):
            return np.full_like(means, float(sd))
        return np.asarray(sd, dtype=float).reshape(means.shape)

    def _draw(self, means, sds):
        means = np.asarray(means, dtype=float)
        sds = np.asarray(sds, dtype=float)
        flat = random_normal_adv(means.flatten().tolist(), sds.flatten().tolist())
        return np.array(flat, dtype=float).reshape(means.shape)

    def _out_of_bounds(self, values, ref):
        if ref is None:
            metric, active = values, np.ones(values.shape, dtype=bool)
        else:
            with np.errstate(divide="ignore", invalid="ignore"):
                metric = np.where(ref != 0, values / ref, 0.0)
            active = ref != 0  # no-electorate cells are trivially fine at 0, skip the gate

        bad = np.zeros(values.shape, dtype=bool)
        if self.low is not None:
            bad |= metric < self.low
        if self.high is not None:
            bad |= metric > self.high
        return bad & active

    def _gate_and_redo(self, values, means, sds, ref):
        for attempt in range(1, self.max_attempts + 1):
            bad = self._out_of_bounds(values, ref)
            if not bad.any():
                return values, attempt, 0
            values[bad] = self._draw(means[bad], sds[bad])
        return values, self.max_attempts, int(self._out_of_bounds(values, ref).sum())


In [30]:
def rand_col(mean, ref, name, sd_fraction=0.1, low=0, high=None, attempts=3):
    values, _ = GaussianRandomizer(sd_fraction, low, high, attempts).randomize(
        mean.to_frame(name), reference=ref.to_frame(name), clamp_to_reference=True
    )
    return values[name]

def reconcile3(draws, total):
    order = np.argsort(draws, axis=0)
    lo, mid, hi = np.take_along_axis(draws, order, axis=0)
    mid = np.minimum(mid, total - lo)
    hi = total - lo - mid
    final = np.empty_like(draws)
    np.put_along_axis(final, order, np.stack([lo, mid, hi]), axis=0)
    return final

df_general_direct["Turnout"] = rand_col(df_general_direct.Turnout_Model * df_general_direct.Electorate, df_general_direct.Electorate, "Turnout", sd_fraction=0.05, high=0.975).round().astype(int)
df_general_direct["Spoil"] = rand_col(df_general_direct.Spoil_Model * df_general_direct.Turnout, df_general_direct.Turnout, "Spoil", sd_fraction=0.035, low=0.002).round().astype(int)
df_general_direct["Valid_Votes"] = df_general_direct.Turnout - df_general_direct.Spoil

vv = df_general_direct.Valid_Votes
draws = np.stack([rand_col(df_general_direct[p] * vv, vv, p, sd_fraction=0.035).to_numpy() for p in ["L", "MR", "R"]])
df_general_direct[["L_Votes", "MR_Votes", "R_Votes"]] = reconcile3(draws, vv.to_numpy()).T.astype(int)

assert (df_general_direct.L_Votes + df_general_direct.MR_Votes + df_general_direct.R_Votes + df_general_direct.Spoil == df_general_direct.Turnout).all()

for votes_col, prefix in [("L_Votes", "Left"), ("MR_Votes", "MR"), ("R_Votes", "Right")]:
    dests = [f"{prefix}_Left", f"{prefix}_Right", f"{prefix}_MR"]
    src = df_general_direct[votes_col]
    draws = np.stack([rand_col(df_general_direct[d] * src, src, d).to_numpy() for d in dests])
    df_general_direct[dests] = reconcile3(draws, src.to_numpy()).T.astype(int)
    assert (df_general_direct[dests].sum(axis=1) == src).all()

df_general_direct


E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding 

,Constituency,Sub_Constituency,Electorate,Turnout_Model,Spoil_Model,L,MR,R,Left_Left,Left_Right,...,Right_MR,MR_Left,MR_Right,MR_MR,Turnout,Spoil,Valid_Votes,L_Votes,MR_Votes,R_Votes
0,Planning and Research,Planning and Research,14342,0.950,0.010,0.462537,0.518929,0.018534,3358,575,...,127,1758,373,4794,13407,136,13271,6110,6925,236
1,Applied Financial Maths,Financial Risk Management,12000,0.825,0.062,0.640736,0.300037,0.059228,3430,884,...,240,1243,957,704,9676,610,9066,5613,2904,549
2,Applied Financial Maths,Time Series Analysis,12000,0.755,0.096,0.659321,0.243001,0.097677,4273,528,...,202,793,781,450,9261,903,8358,5594,2024,740
3,Bayesian Modelling and Statistical Machine Lea...,Bayesian Modelling,12000,0.860,0.045,0.532692,0.431795,0.035513,2975,621,...,148,1635,336,2089,9834,427,9407,5002,4060,345
4,Bayesian Modelling and Statistical Machine Lea...,Statistical Machine Learning,12000,0.900,0.029,0.442175,0.519682,0.038143,3286,313,...,225,1605,502,3151,10512,300,10212,4540,5258,414
5,Coding and Applied Data Science,Mathematics Project,12000,0.925,0.018,0.399075,0.568815,0.032110,2432,370,...,231,1360,572,4278,11224,188,11036,4461,6210,365
6,Coding and Applied Data Science,Programming and Data Analysis,12000,0.880,0.021,0.376079,0.580393,0.043528,1818,615,...,275,1013,547,4126,10023,220,9803,3690,5686,427
7,Diplomatic Team,Diplomats Rep,7323,0.830,0.012,0.308069,0.383030,0.308901,982,329,...,824,727,655,971,6119,70,6049,1849,2353,1847
8,Diplomatic Team,Academic and Project Reps,17366,0.925,0.008,0.410353,0.573958,0.015689,4758,370,...,113,2913,654,5005,15601,124,15477,6664,8572,241
9,Finance Management,Finance Management,27531,0.870,0.043,0.387051,0.196319,0.416630,2834,1121,...,3169,2393,749,1349,23647,1023,22624,8874,4491,9259


In [31]:
df_general_direct_output = df_general_direct[[
    "Constituency", "Sub_Constituency", "Electorate", "Turnout",
    "Left_Left", "Left_Right", "Left_MR",
    "Right_Left", "Right_Right", "Right_MR",
    "MR_Left", "MR_Right", "MR_MR",
]].copy()
df_general_direct_output[["Spoil", "received_at", "verified_at", "declared_at"]] = ""



In [32]:
# df_general_direct.to_csv("./generated_result/test_df_results_generaldirect.csv", index=False)

## Build `df_generaldirect_referendum1`

Same election, same voters as `df_general_direct` — `Turnout` is copied straight across (joined on `Constituency` + `Sub_Constituency`, which match this table 1:1) rather than re-randomized. `Spoil` and the `Yes_Votes`/`No_Votes` split reuse the same `rand_col` technique used for direct.

In [33]:
df_generaldirect_referendum1 = df_generaldirect_referendum1.merge(
    df_general_direct[["Constituency", "Sub_Constituency", "Turnout"]].rename(columns={"Turnout": "Turnout_Direct"}),
    on=["Constituency", "Sub_Constituency"], how="left",
)
assert df_generaldirect_referendum1["Turnout_Direct"].notna().all()
df_generaldirect_referendum1["Turnout"] = df_generaldirect_referendum1.pop("Turnout_Direct").astype(int)

referendum1_spoil_model = df_generaldirect_referendum1["Spoil"]
df_generaldirect_referendum1["Spoil"] = rand_col(referendum1_spoil_model * df_generaldirect_referendum1.Turnout, df_generaldirect_referendum1.Turnout, "Spoil", sd_fraction=0.035, low=0.002).round().astype(int)
df_generaldirect_referendum1["Valid_Votes"] = df_generaldirect_referendum1.Turnout - df_generaldirect_referendum1.Spoil

vv1 = df_generaldirect_referendum1.Valid_Votes
yes_model = df_generaldirect_referendum1["Yes_Votes"]
df_generaldirect_referendum1["Yes_Votes"] = rand_col(yes_model * vv1, vv1, "Yes_Votes", sd_fraction=0.035).round().astype(int)
df_generaldirect_referendum1["No_Votes"] = vv1 - df_generaldirect_referendum1["Yes_Votes"]

assert (df_generaldirect_referendum1.Yes_Votes + df_generaldirect_referendum1.No_Votes + df_generaldirect_referendum1.Spoil == df_generaldirect_referendum1.Turnout).all()

df_results_generaldirect_referendum1 = df_generaldirect_referendum1[[
    "Constituency", "Sub_Constituency", "Electorate", "Turnout",
    "Yes_Votes", "No_Votes", "Spoil",
]].copy()
df_results_generaldirect_referendum1[["received_at", "verified_at", "declared_at"]] = ""



E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


## Build `df_general_proportional`

`Turnout` is only copied in where a row has a real counterpart in `df_general_direct` — the 8 constituency-level rows, filled by summing `df_general_direct.Turnout` across their direct sub-constituencies. The 6 `Swift` rows are a finer split that doesn't exist in `df_general_direct`, so those (already filled) Turnout values are left untouched. `Spoil` and the `NRD`/`LRP`/`GRF`/`GCCA` split reuse the direct table's technique, generalized from `reconcile3` to N categories via `reconcileN`.

In [34]:
def reconcileN(draws, total):
    order = np.argsort(draws, axis=0)
    sorted_draws = np.take_along_axis(draws, order, axis=0)
    remaining = total.copy()
    allocated = np.empty_like(sorted_draws)
    for i in range(sorted_draws.shape[0] - 1):
        allocated[i] = np.minimum(sorted_draws[i], remaining)
        remaining = remaining - allocated[i]
    allocated[-1] = remaining
    final = np.empty_like(draws)
    np.put_along_axis(final, order, allocated, axis=0)
    return final

In [35]:
turnout_by_constituency = df_general_direct.groupby("Constituency")["Turnout"].sum()
missing_turnout = df_general_proportional["Turnout"].isna()
df_general_proportional.loc[missing_turnout, "Turnout"] = (
    df_general_proportional.loc[missing_turnout, "Constituency"].map(turnout_by_constituency)
)
assert df_general_proportional["Turnout"].notna().all()
df_general_proportional["Turnout"] = df_general_proportional["Turnout"].astype(int)

proportional_spoil_model = df_general_proportional["Spoil"]
df_general_proportional["Spoil"] = rand_col(proportional_spoil_model * df_general_proportional.Turnout, df_general_proportional.Turnout, "Spoil", sd_fraction=0.035, low=0.002).round().astype(int)
df_general_proportional["Valid_Votes"] = df_general_proportional.Turnout - df_general_proportional.Spoil

vv2 = df_general_proportional.Valid_Votes
parties = ["NRD", "LRP", "GRF", "GCCA"]
draws = np.stack([rand_col(df_general_proportional[p] * vv2, vv2, p, sd_fraction=0.035).to_numpy() for p in parties])
df_general_proportional[parties] = reconcileN(draws, vv2.to_numpy()).T.astype(int)

assert (df_general_proportional[parties].sum(axis=1) == vv2).all()

df_general_proportional_output = df_general_proportional[[
    "Constituency", "Sub_Constituency", "Electorate", "Turnout",
    "NRD", "LRP", "GRF", "GCCA", "Spoil",
]].copy()
df_general_proportional_output[["received_at", "verified_at", "declared_at"]] = ""

df_general_proportional_output

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


,Constituency,Sub_Constituency,Electorate,Turnout,NRD,LRP,GRF,GCCA,Spoil,received_at,verified_at,declared_at
0,Planning and Research,Planning and Research,14342,13407,5291,977,6753,287,99,,,
1,Applied Financial Maths,Applied Financial Maths,24000,18937,7852,3537,4763,1598,1187,,,
2,Bayesian Modelling and Statistical Machine Lea...,Bayesian Modelling and Statistical Machine Lea...,24000,20346,8014,2021,8947,934,430,,,
3,Coding and Applied Data Science,Coding and Applied Data Science,24000,21247,4806,3100,11894,1135,312,,,
4,Diplomatic Team,Diplomatic Team,24689,21720,2639,6203,10521,2174,183,,,
5,Finance Management,Finance Management,27531,23647,2064,8260,3853,9048,422,,,
6,Decision and Execution,Decision and Execution,9147,8465,690,1920,3727,2035,93,,,
7,Campaign and Career Team,Campaign and Career Team,17716,16385,5659,2175,7491,803,257,,,
8,Swift,Swift (Outland),730,631,131,15,62,374,49,,,
9,Swift,Swift 1,15559,13533,5784,1800,4988,760,201,,,


In [36]:
df_general_proportional_output.to_csv("./generated_result/test_df_results_generalproportional.csv", index=False)

## Build `df_13d`

Two-party race — no MR line exists at all, so the `Left`/`Right` split is the 2-category case (same as the referendum `Yes`/`No` step): draw `Left` via `rand_col`, `Right` is the remainder. Since there's no MR to redistribute votes into, everything stays on the diagonal — `Left_Left = Left`, `Right_Right = Right`, and the six MR-related cross-tab columns are just `0`.

In [37]:
df_13d["Turnout"] = rand_col(df_13d["Turnout Model"] * df_13d.Electorate, df_13d.Electorate, "Turnout", sd_fraction=0.05, high=0.975).round().astype(int)
df_13d["Spoil"] = rand_col(df_13d["Spoil Data Model"] * df_13d.Turnout, df_13d.Turnout, "Spoil", sd_fraction=0.025, low=0.002).round().astype(int)
df_13d["Valid_Votes"] = df_13d.Turnout - df_13d.Spoil

vv3 = df_13d.Valid_Votes
df_13d["Left"] = rand_col(df_13d.Left_Model * vv3, vv3, "Left", sd_fraction=0.04).round().astype(int)
df_13d["Right"] = vv3 - df_13d["Left"]

assert (df_13d.Left + df_13d.Right + df_13d.Spoil == df_13d.Turnout).all()

df_13d["Left_Left"] = df_13d["Left"]
df_13d["Right_Right"] = df_13d["Right"]
for col in ["Left_Right", "Left_MR", "Right_Left", "Right_MR", "MR_Left", "MR_Right", "MR_MR"]:
    df_13d[col] = 0

df_13d_output = df_13d[[
    "Constituency", "Sub_Constituency", "Electorate", "Turnout",
    "Left_Left", "Left_Right", "Left_MR",
    "Right_Left", "Right_Right", "Right_MR",
    "MR_Left", "MR_Right", "MR_MR",
    "Spoil",
]].copy()
df_13d_output[["received_at", "verified_at", "declared_at"]] = ""

df_13d_output

E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)
E:\Coding Site\just_for_fun\random_no_generation.py:27: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_FILE, header=None)


,Constituency,Sub_Constituency,Electorate,Turnout,Left_Left,Left_Right,Left_MR,Right_Left,Right_Right,Right_MR,MR_Left,MR_Right,MR_MR,Spoil,received_at,verified_at,declared_at
0,Shek East and Central,Shek East and Central,675100000,524658885,351145066,0,0,0,163573207,0,0,0,0,9940612,,,
1,"Shek West, Rainbow N, LF & Huang",Shek West,393600000,278282961,155737830,0,0,0,115528041,0,0,0,0,7017090,,,
2,"Shek West, Rainbow N, LF & Huang",Mid and North Rainbow,228200000,149296322,61793473,0,0,0,83356073,0,0,0,0,4146776,,,
3,"Shek West, Rainbow N, LF & Huang",LF,42800000,26905001,8876337,0,0,0,17232292,0,0,0,0,796372,,,
4,"Shek West, Rainbow N, LF & Huang",Huang,56500000,41954866,30363773,0,0,0,11149870,0,0,0,0,441223,,,
5,Tong,Tong Central,14800000,9155907,3483620,0,0,0,5133448,0,0,0,0,538839,,,
6,Tong,Mid Tong,57500000,35720323,12026199,0,0,0,22120290,0,0,0,0,1573834,,,
7,Tong,Upper Tong,8500000,4447208,1847560,0,0,0,2350411,0,0,0,0,249237,,,
8,Diamond & Rainbow SE,Diamond,530000000,388259181,246502952,0,0,0,136858664,0,0,0,0,4897565,,,
9,Diamond & Rainbow SE,Rainbow Southwest,135000000,101978916,67511827,0,0,0,32816904,0,0,0,0,1650185,,,


In [39]:
df_13d_output.to_csv("./generated_result/df_results_13D_pending_verification.csv", index=False)
df_general_direct_output.to_csv("./generated_result/df_results_general_direct_pending_verification.csv", index=False)
df_results_generaldirect_referendum1.to_csv("./generated_result/df_results_generaldirect_referendum1_pending_verification.csv", index=False)
df_general_proportional_output.to_csv("./generated_result/df_results_general_proportional_pending_verification.csv", index=False)